# Classification de gestes sportifs avec PyTorch
## Tutoriel 4 / 4

On entraîne un réseau de neurones PyTorch à distinguer des gestes sportifs
à partir des paramètres SMPL extraits dans les tutoriels précédents.

**Exemple : Tennis — Coup droit vs Revers**

```
Vidéos tennis (coup droit)  ┐
                             ├→ MediaPipe → SMPL → Features → PyTorch MLP → Label
Vidéos tennis (revers)      ┘
```

**Ce qu'on apprend :**
- Construire un dataset PyTorch depuis des fichiers SMPL
- Extraire des features pertinentes depuis des séquences de poses
- Entraîner un MLP + un LSTM et comparer les deux
- Évaluer avec matrice de confusion

In [ ]:
!pip install mediapipe opencv-python-headless matplotlib yt-dlp --quiet

import cv2, mediapipe as mp, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from scipy.spatial.transform import Rotation as sRot
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import warnings; warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

mp_pose = mp.solutions.pose
MP = mp_pose.PoseLandmark

## 1. Téléchargement des vidéos par classe

On définit un dictionnaire `CLASSES` : nom de classe → liste de mots-clés YouTube.
Chaque vidéo téléchargée devient un exemple d'entraînement.

In [ ]:
import yt_dlp, re

# ── À MODIFIER selon votre problème ─────────────────────────────────────────
CLASSES = {
    "forehand": [
        "tennis forehand slow motion technique",
        "tennis forehand tutorial side view",
        "forehand tennis stroke slow motion",
    ],
    "backhand": [
        "tennis backhand slow motion technique",
        "tennis backhand tutorial side view",
        "backhand tennis stroke slow motion",
    ],
}
MAX_PER_QUERY   = 4     # vidéos par requête
MAX_DURATION    = 120   # secondes
MIN_DURATION    = 8
VIDEO_ROOT      = Path("videos_tennis")
# ────────────────────────────────────────────────────────────────────────────

def search_and_download(query, out_dir, max_results=4, max_dur=120, min_dur=8):
    """Recherche et télécharge des vidéos YouTube filtrées par durée."""
    out_dir.mkdir(parents=True, exist_ok=True)
    search_url = f"ytsearch{max_results*2}:{query}"

    with yt_dlp.YoutubeDL({"quiet": True, "extract_flat": True}) as ydl:
        info = ydl.extract_info(search_url, download=False)

    downloaded = []
    for entry in (info.get("entries") or []):
        if len(downloaded) >= max_results:
            break
        dur = entry.get("duration") or 0
        if not (min_dur <= dur <= max_dur):
            continue

        title    = re.sub(r'[\\/*?":.<>|]', "", entry.get("title", "video"))[:50]
        out_tmpl = str(out_dir / f"{title}.%(ext)s")
        url      = f"https://www.youtube.com/watch?v={entry['id']}"

        if list(out_dir.glob(f"{title}.*")):
            print(f"  [SKIP] {title[:40]}")
            downloaded.append(list(out_dir.glob(f"{title}.*"))[0])
            continue

        opts = {"quiet": True, "format": "best[height<=480][ext=mp4]/best[ext=mp4]/best",
                "outtmpl": out_tmpl, "merge_output_format": "mp4"}
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.download([url])
            files = list(out_dir.glob(f"{title}.*"))
            if files:
                print(f"  [OK] {title[:40]}")
                downloaded.append(files[0])
        except Exception as e:
            print(f"  [ERR] {str(e)[:60]}")

    return downloaded


# Téléchargement par classe
class_files = {}   # classe → liste de chemins vidéo
for cls, queries in CLASSES.items():
    out_dir = VIDEO_ROOT / cls
    files   = []
    for q in queries:
        print(f"\n[{cls}] '{q}'")
        files += search_and_download(q, out_dir, MAX_PER_QUERY, MAX_DURATION, MIN_DURATION)
    # Dédoublonnage
    class_files[cls] = list({str(f): f for f in files}.values())
    print(f"  → {len(class_files[cls])} vidéos pour '{cls}'")

print("\nDataset brut :")
for cls, files in class_files.items():
    print(f"  {cls:12s} : {len(files)} vidéos")

## 2. Extraction des features SMPL

On utilise les fonctions du tutoriel 2 pour convertir chaque vidéo
en paramètres SMPL, puis on extrait des **features statistiques** sur la séquence.

Chaque vidéo → un vecteur fixe de features → un exemple d'entraînement.

In [ ]:
# ── Reprise des fonctions du tutoriel 2 ─────────────────────────────────────

SMPL_BONES = {
    "L_Hip":   (MP.LEFT_HIP,   MP.LEFT_KNEE),
    "L_Knee":  (MP.LEFT_KNEE,  MP.LEFT_ANKLE),
    "R_Hip":   (MP.RIGHT_HIP,  MP.RIGHT_KNEE),
    "R_Knee":  (MP.RIGHT_KNEE, MP.RIGHT_ANKLE),
    "L_Shoulder": (MP.LEFT_SHOULDER,  MP.LEFT_ELBOW),
    "L_Elbow":    (MP.LEFT_ELBOW,     MP.LEFT_WRIST),
    "R_Shoulder": (MP.RIGHT_SHOULDER, MP.RIGHT_ELBOW),
    "R_Elbow":    (MP.RIGHT_ELBOW,    MP.RIGHT_WRIST),
    "Chest":      (MP.LEFT_SHOULDER,  MP.RIGHT_SHOULDER),
}

T_POSE_DIRS = {
    "L_Hip": np.array([0,-1,0]), "L_Knee": np.array([0,-1,0]),
    "R_Hip": np.array([0,-1,0]), "R_Knee": np.array([0,-1,0]),
    "L_Shoulder": np.array([-1,0,0]), "L_Elbow": np.array([-1,0,0]),
    "R_Shoulder": np.array([ 1,0,0]), "R_Elbow": np.array([ 1,0,0]),
    "Chest": np.array([1,0,0]),
}

def vec_to_vec_rotation(v_from, v_to):
    v_from = v_from / (np.linalg.norm(v_from) + 1e-9)
    v_to   = v_to   / (np.linalg.norm(v_to)   + 1e-9)
    dot    = np.clip(np.dot(v_from, v_to), -1., 1.)
    angle  = np.arccos(dot)
    if angle < 1e-6: return np.zeros(3)
    axis   = np.cross(v_from, v_to)
    n      = np.linalg.norm(axis)
    if n < 1e-9: axis = np.array([1,0,0])
    else: axis = axis / n
    return axis * angle


def video_to_pose_sequence(video_path, max_frames=150):
    """
    Extrait une séquence de rotations articulaires depuis une vidéo.
    Retourne (T, N_bones * 3) array.
    """
    bone_names = list(SMPL_BONES.keys())  # N_bones = 9
    sequence   = []

    cap = cv2.VideoCapture(str(video_path))
    with mp_pose.Pose(model_complexity=1, smooth_landmarks=True,
                      min_detection_confidence=0.5,
                      min_tracking_confidence=0.5) as pose:
        n = 0
        while cap.isOpened() and n < max_frames:
            ret, frame = cap.read()
            if not ret: break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)
            if res.pose_world_landmarks:
                lm  = res.pose_world_landmarks.landmark
                xyz = np.array([[l.x, l.y, l.z] for l in lm])
                row = []
                for bn in bone_names:
                    pi, di = SMPL_BONES[bn]
                    bone_v = xyz[di.value] - xyz[pi.value]
                    row.append(vec_to_vec_rotation(T_POSE_DIRS[bn], bone_v))
                sequence.append(np.concatenate(row))  # (27,)
            else:
                if sequence: sequence.append(sequence[-1])  # repeat last
                else: sequence.append(np.zeros(len(bone_names) * 3))
            n += 1
    cap.release()
    return np.array(sequence)  # (T, 27)


def extract_features(sequence):
    """
    Résume une séquence (T, D) en un vecteur fixe de features statistiques.
    Features : mean, std, min, max, range par dimension = 5 × D valeurs.
    """
    if len(sequence) == 0:
        return np.zeros(5 * 27)
    mean  = sequence.mean(axis=0)
    std   = sequence.std(axis=0)
    vmin  = sequence.min(axis=0)
    vmax  = sequence.max(axis=0)
    rng   = vmax - vmin
    return np.concatenate([mean, std, vmin, vmax, rng])  # (135,)


print("Fonctions prêtes.")

In [ ]:
# Extraction des features pour toutes les vidéos
LABEL_MAP = {cls: i for i, cls in enumerate(CLASSES.keys())}
print("Labels :", LABEL_MAP)

X_all, y_all, names_all = [], [], []
sequences_all = []   # pour le LSTM
SEQ_LEN = 60         # longueur fixe pour le LSTM (frames)

for cls, files in class_files.items():
    label = LABEL_MAP[cls]
    for fpath in files:
        print(f"  [{cls}] {fpath.name[:50]}…", end=" ")
        seq = video_to_pose_sequence(str(fpath), max_frames=SEQ_LEN)
        if len(seq) < 10:
            print("trop court, ignoré")
            continue

        # Padder / tronquer à SEQ_LEN pour le LSTM
        if len(seq) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - len(seq), seq.shape[1]))
            seq_fixed = np.vstack([seq, pad])
        else:
            seq_fixed = seq[:SEQ_LEN]

        feat = extract_features(seq)
        X_all.append(feat)
        sequences_all.append(seq_fixed)
        y_all.append(label)
        names_all.append(fpath.name)
        print(f"{len(seq)} frames → features {feat.shape}")

X = np.array(X_all)           # (N, 135) pour MLP
S = np.array(sequences_all)   # (N, SEQ_LEN, 27) pour LSTM
y = np.array(y_all)

print(f"\nDataset : {len(y)} exemples  ({np.bincount(y)})")
print(f"  MLP features : {X.shape}")
print(f"  LSTM sequences : {S.shape}")

## 3. Dataset et DataLoader PyTorch

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split train / test
idx = np.arange(len(y))
idx_train, idx_test = train_test_split(idx, test_size=0.25,
                                       stratify=y, random_state=42)

# Normalisation des features MLP
scaler = StandardScaler().fit(X[idx_train])
X_norm = scaler.transform(X)

class PoseDataset(Dataset):
    def __init__(self, features, sequences, labels):
        self.X = torch.tensor(features,  dtype=torch.float32)
        self.S = torch.tensor(sequences, dtype=torch.float32)
        self.y = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.S[i], self.y[i]

train_ds = PoseDataset(X_norm[idx_train], S[idx_train], y[idx_train])
test_ds  = PoseDataset(X_norm[idx_test],  S[idx_test],  y[idx_test])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False)

N_CLASSES  = len(CLASSES)
FEAT_DIM   = X.shape[1]
SEQ_FEAT   = S.shape[2]

print(f"Train : {len(train_ds)}   Test : {len(test_ds)}")
print(f"Feature dim MLP : {FEAT_DIM}   Sequence dim LSTM : {SEQ_FEAT}")

## 4. Modèle A — MLP (features statistiques)

In [ ]:
class PoseMLP(nn.Module):
    """
    Réseau dense simple sur les features statistiques SMPL.
    Input : (batch, 135)  →  Output : (batch, N_classes)
    """
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),     nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x, s=None):   # s ignoré
        return self.net(x)


def train_model(model, loader_train, loader_test, n_epochs=50):
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_epochs)
    loss_fn = nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(n_epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for xb, sb, yb in loader_train:
            xb, sb, yb = xb.to(DEVICE), sb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb, sb)
            loss   = loss_fn(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            total      += len(yb)
        sched.step()

        # Évaluation
        model.eval()
        correct_t, total_t = 0, 0
        with torch.no_grad():
            for xb, sb, yb in loader_test:
                xb, sb, yb = xb.to(DEVICE), sb.to(DEVICE), yb.to(DEVICE)
                correct_t += (model(xb, sb).argmax(1) == yb).sum().item()
                total_t   += len(yb)

        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(correct / total)
        history["test_acc"].append(correct_t / total_t)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs}  "
                  f"loss={history['train_loss'][-1]:.3f}  "
                  f"train_acc={history['train_acc'][-1]:.2%}  "
                  f"test_acc={history['test_acc'][-1]:.2%}")

    return model, history


print("Entraînement MLP...")
mlp = PoseMLP(FEAT_DIM, N_CLASSES)
mlp, hist_mlp = train_model(mlp, train_loader, test_loader, n_epochs=60)

## 5. Modèle B — LSTM (séquence temporelle)

In [ ]:
class PoseLSTM(nn.Module):
    """
    LSTM sur la séquence de poses.
    Input : (batch, T, D)  →  Output : (batch, N_classes)
    """
    def __init__(self, in_dim, hidden=128, n_layers=2, n_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, n_layers,
                            batch_first=True, dropout=0.3)
        self.head = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x, s):   # x = features MLP (ignoré), s = séquence
        _, (h, _) = self.lstm(s)   # h : (n_layers, batch, hidden)
        return self.head(h[-1])    # dernière couche


print("Entraînement LSTM...")
lstm = PoseLSTM(SEQ_FEAT, hidden=128, n_layers=2, n_classes=N_CLASSES)
lstm, hist_lstm = train_model(lstm, train_loader, test_loader, n_epochs=60)

## 6. Comparaison et évaluation

In [ ]:
# Courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs = range(1, 61)

for ax, hist, name in zip(axes, [hist_mlp, hist_lstm], ["MLP", "LSTM"]):
    ax.plot(epochs, hist["train_acc"], label="Train", lw=2)
    ax.plot(epochs, hist["test_acc"],  label="Test",  lw=2, linestyle="--")
    ax.set_title(f"{name} — Accuracy")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"MLP  final test acc : {hist_mlp['test_acc'][-1]:.2%}")
print(f"LSTM final test acc : {hist_lstm['test_acc'][-1]:.2%}")

In [ ]:
def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, sb, yb in loader:
            xb, sb = xb.to(DEVICE), sb.to(DEVICE)
            logits = model(xb, sb)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(trues), np.array(preds)


CLASS_NAMES = list(CLASSES.keys())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, name in zip(axes, [mlp, lstm], ["MLP", "LSTM"]):
    y_true, y_pred = get_predictions(model, test_loader)
    cm = confusion_matrix(y_true, y_pred)

    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
    ax.set_title(f"Matrice de confusion — {name}")
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
    plt.colorbar(im, ax=ax)

plt.tight_layout(); plt.show()

# Rapport détaillé
print("\n── MLP ──")
y_true, y_pred = get_predictions(mlp, test_loader)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

print("── LSTM ──")
y_true, y_pred = get_predictions(lstm, test_loader)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 7. Inférence sur une nouvelle vidéo

In [ ]:
def predict_gesture(video_path, model, scaler, use_lstm=False):
    """Prédit la classe d'un geste dans une vidéo."""
    seq  = video_to_pose_sequence(str(video_path), max_frames=SEQ_LEN)

    # Features MLP
    feat   = extract_features(seq)
    feat_n = scaler.transform(feat.reshape(1, -1))
    x_t    = torch.tensor(feat_n, dtype=torch.float32).to(DEVICE)

    # Séquence LSTM
    if len(seq) < SEQ_LEN:
        pad = np.zeros((SEQ_LEN - len(seq), seq.shape[1]))
        seq = np.vstack([seq, pad])
    else:
        seq = seq[:SEQ_LEN]
    s_t = torch.tensor(seq[None], dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x_t, s_t)
        probs  = torch.softmax(logits, dim=-1)[0]
        pred   = probs.argmax().item()

    print(f"Vidéo : {Path(video_path).name}")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"  {cls:12s} : {probs[i].item():.1%}")
    print(f"  → Prédiction : {CLASS_NAMES[pred].upper()}")
    return CLASS_NAMES[pred]


# Test sur les vidéos de test
print("Prédictions sur quelques exemples de test :\n")
for i in idx_test[:4]:
    # Récupérer le chemin depuis names_all
    # (chercher la vidéo dans le dataset)
    cls   = CLASS_NAMES[y[i]]
    fpath = class_files[cls][0]  # exemple
    predict_gesture(fpath, lstm, scaler, use_lstm=True)
    print()

## Récapitulatif du pipeline complet

```
01_download_videos.ipynb
    └─ yt-dlp : mots-clés → vidéos MP4

02_mediapipe_to_smpl.ipynb
    └─ MediaPipe : vidéo → 33 landmarks 3D
    └─ Mapping géométrique : landmarks → body_pose SMPL (69 valeurs/frame)

03_mediapipe_pose.ipynb
    └─ Visualisation, analyse temporelle, détection de cycle

04_classification.ipynb  ← CE NOTEBOOK
    └─ Features statistiques (mean/std/range) sur la séquence SMPL
    └─ MLP  : features → classe
    └─ LSTM : séquence → classe
    └─ Matrice de confusion + rapport de classification
```

**Pour aller plus loin** : remplacer le mapping MediaPipe→SMPL par GVHMR
donnera des features plus précises et donc une meilleure classification.